<a href="https://colab.research.google.com/github/aishi10/CPI_Project/blob/main/part2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

!pip -q install -U transformers accelerate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 66.9 MB/s eta 0:00:00


In [2]:

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto"
)

model.eval()

print("Model loaded successfully!")
print("PyTorch version:", torch.__version__)


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!
PyTorch version: 2.11.0+cu130


In [3]:

import json

memory_content = "The user prefers Python over Java"
new_information = "The user now prefers Java over Python"

prompt = f"""
You are an evidence classifier in a memory revision system.

EXISTING MEMORY:
{memory_content}

NEW INFORMATION:
{new_information}

Classify the relationship between the new information and the memory.

Allowed relations:
- support: evidence in favor of the memory
- contradiction: evidence against the memory
- neither: neither supports nor contradicts the memory

Strength must be between 0.0 and 1.0.
For neither, strength must be 0.0.
Do not revise the memory or recommend an action.

Return only a JSON object in this format:
{{"relation": "contradiction", "strength": 0.9}}
"""

messages = [
    {
        "role": "user",
        "content": prompt
    }
]

text_input = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text_input,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False
    )

generated_tokens = outputs[
    0, inputs["input_ids"].shape[1]:
]

response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print("LLM response:")
print(response)


LLM response:
{
  "relation": "contradiction",
  "strength": 0.9
}


In [4]:

def parse_and_validate(response):
    result = json.loads(response.strip())

    if not isinstance(result, dict):
        raise ValueError("Response must be a JSON object")

    if set(result.keys()) != {"relation", "strength"}:
        raise ValueError("Unexpected or missing fields")

    if result["relation"] not in {
        "support", "contradiction", "neither"
    }:
        raise ValueError("Invalid relation")

    strength = result["strength"]

    if (
        isinstance(strength, bool)
        or not isinstance(strength, (int, float))
        or not 0.0 <= strength <= 1.0
    ):
        raise ValueError("Strength must be between 0 and 1")

    if result["relation"] == "neither" and strength != 0.0:
        raise ValueError("Neither must have strength 0.0")

    return result


try:
    classification = parse_and_validate(response)
    print("Validated classification:", classification)
except (ValueError, TypeError, json.JSONDecodeError) as error:
    print("Response needs correction:", error)


Validated classification: {'relation': 'contradiction', 'strength': 0.9}


In [5]:

def classify_evidence(memory_content, new_information):
    prompt = f"""
You are an evidence classifier in a memory revision system.

EXISTING MEMORY:
{memory_content}

NEW INFORMATION:
{new_information}

Choose exactly one relation:
- support: evidence in favor of the memory
- contradiction: evidence against the memory
- neither: neither supports nor contradicts the memory

Strength must be between 0.0 and 1.0.
For neither, strength must be 0.0.
Do not revise the memory or recommend an action.
Do not assume facts that are not provided.

Return only a JSON object:
{{"relation": "support", "strength": 0.8}}
"""

    messages = [{"role": "user", "content": prompt}]

    text_input = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text_input,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False
        )

    generated_tokens = outputs[
        0, inputs["input_ids"].shape[1]:
    ]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return parse_and_validate(response)


In [6]:

test_cases = [
    (
        "The user prefers Python over Java",
        "The user now prefers Java over Python"
    ),
    (
        "The library closes at 6 PM",
        "The official library website confirms it closes at 6 PM"
    ),
    (
        "The user prefers Python over Java",
        "The user bought a new laptop yesterday"
    )
]

for memory, new_info in test_cases:
    result = classify_evidence(memory, new_info)

    print("\nMemory:", memory)
    print("New information:", new_info)
    print("LLM result:", result)



Memory: The user prefers Python over Java
New information: The user now prefers Java over Python
LLM result: {'relation': 'support', 'strength': 0.8}

Memory: The library closes at 6 PM
New information: The official library website confirms it closes at 6 PM
LLM result: {'relation': 'support', 'strength': 0.8}

Memory: The user prefers Python over Java
New information: The user bought a new laptop yesterday
LLM result: {'relation': 'support', 'strength': 0.8}


In [7]:

def classify_evidence(memory_content, new_information):
    prompt = f"""
You are a careful evidence classifier.

Your task is to compare an existing memory with new information.
Classify ONLY the logical relationship between them.

RELATIONS:
- support: The new information makes the existing memory more likely to be true.
- contradiction: The new information makes the existing memory less likely to be true.
- neither: The new information does not meaningfully affect the truth of the memory.

EXAMPLES:

Memory: The user prefers Python over Java
New information: The user enjoys using Python more than Java
Output: {{"relation": "support", "strength": 0.9}}

Memory: The user prefers Python over Java
New information: The user now prefers Java over Python
Output: {{"relation": "contradiction", "strength": 0.95}}

Memory: The user prefers Python over Java
New information: The user bought a new laptop yesterday
Output: {{"relation": "neither", "strength": 0.0}}

Memory: The user prefers Python over Java
New information: The user is learning Java for a job
Output: {{"relation": "neither", "strength": 0.0}}

Memory: The library closes at 6 PM
New information: The official library website confirms it closes at 6 PM
Output: {{"relation": "support", "strength": 0.95}}

NOW CLASSIFY THIS CASE.

Memory: {memory_content}
New information: {new_information}

Rules:
1. Identify what the memory actually claims.
2. Check whether the new information supports, contradicts, or is unrelated to that claim.
3. Learning a skill does not necessarily mean preferring it.
4. Buying something or performing an unrelated activity is not evidence of a preference.
5. Do not always choose support. Choose the relation justified by the facts.
6. If neither, strength MUST be 0.0.
7. Return ONLY a JSON object with keys relation and strength.
8. relation must be support, contradiction, or neither.
9. strength must be a number from 0.0 to 1.0.

Output:
"""

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text_input = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text_input,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False
        )

    generated_tokens = outputs[
        0, inputs["input_ids"].shape[1]:
    ]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return parse_and_validate(response)


In [8]:

test_cases = [
    (
        "The user prefers Python over Java",
        "The user now prefers Java over Python"
    ),
    (
        "The library closes at 6 PM",
        "The official library website confirms it closes at 6 PM"
    ),
    (
        "The user prefers Python over Java",
        "The user bought a new laptop yesterday"
    )
]

for memory, new_info in test_cases:
    result = classify_evidence(memory, new_info)
    print("\nMemory:", memory)
    print("New information:", new_info)
    print("LLM result:", result)



Memory: The user prefers Python over Java
New information: The user now prefers Java over Python
LLM result: {'relation': 'neither', 'strength': 0.0}

Memory: The library closes at 6 PM
New information: The official library website confirms it closes at 6 PM
LLM result: {'relation': 'neither', 'strength': 0.0}

Memory: The user prefers Python over Java
New information: The user bought a new laptop yesterday
LLM result: {'relation': 'neither', 'strength': 0.0}


In [9]:

messages = [
    {
        "role": "system",
        "content": "You are a text classification system. Follow the examples and return only JSON."
    },
    {
        "role": "user",
        "content": """
Classify the relationship between each memory and new information.

Example 1:
Memory: The sky is blue.
New information: The sky is blue.
Answer: {"relation": "support", "strength": 0.95}

Example 2:
Memory: The sky is blue.
New information: The sky is not blue; it is green.
Answer: {"relation": "contradiction", "strength": 0.95}

Example 3:
Memory: The sky is blue.
New information: I bought a sandwich.
Answer: {"relation": "neither", "strength": 0.0}

Now classify:
Memory: The user prefers Python over Java.
New information: The user now prefers Java over Python.

Return only the JSON answer.
"""
    }
]

text_input = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text_input,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False
    )

response = tokenizer.decode(
    outputs[0, inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)


{"relation": "neutral", "strength": 0.8}


In [10]:
!pip -q install -U sentence-transformers

from sentence_transformers import CrossEncoder
import numpy as np

nli_model = CrossEncoder("cross-encoder/nli-deberta-v3-small")

print("NLI model loaded successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 34.5 MB/s eta 0:00:00


config.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  568MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

...

spm.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/8.66M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

NLI model loaded successfully!


In [11]:
tests = [
    (
        "The user prefers Python over Java.",
        "The user now prefers Java over Python.",
        "contradiction"
    ),
    (
        "The library closes at 6 PM.",
        "The official website confirms that the library closes at 6 PM.",
        "support"
    ),
    (
        "The user prefers Python over Java.",
        "The user bought a new laptop yesterday.",
        "neither"
    )
]

# NLI order: contradiction, entailment, neutral
labels = ["contradiction", "support", "neither"]

pairs = [(new_info, memory) for memory, new_info, expected in tests]
scores = nli_model.predict(pairs)

for (memory, new_info, expected), row in zip(tests, scores):
    predicted = labels[int(np.argmax(row))]

    print("-" * 60)
    print("Memory:   ", memory)
    print("New info: ", new_info)
    print("Expected: ", expected)
    print("Predicted:", predicted)
    print("Scores:   ", dict(zip(labels, np.round(row, 3))))

------------------------------------------------------------
Memory:    The user prefers Python over Java.
New info:  The user now prefers Java over Python.
Expected:  contradiction
Predicted: contradiction
Scores:    {'contradiction': np.float32(5.276), 'support': np.float32(-2.057), 'neither': np.float32(-3.237)}
------------------------------------------------------------
Memory:    The library closes at 6 PM.
New info:  The official website confirms that the library closes at 6 PM.
Expected:  support
Predicted: support
Scores:    {'contradiction': np.float32(-3.8), 'support': np.float32(4.278), 'neither': np.float32(-0.581)}
------------------------------------------------------------
Memory:    The user prefers Python over Java.
New info:  The user bought a new laptop yesterday.
Expected:  neither
Predicted: neither
Scores:    {'contradiction': np.float32(-2.074), 'support': np.float32(-2.971), 'neither': np.float32(5.72)}


In [12]:
import numpy as np

LABELS = ["contradiction", "support", "neither"]

def classify_evidence(memory_content, new_information):
    # New information is the premise; existing memory is the hypothesis.
    scores = nli_model.predict([(new_information, memory_content)])[0]
    scores = np.asarray(scores, dtype=float)

    # Convert raw scores into probabilities.
    exp_scores = np.exp(scores - np.max(scores))
    probabilities = exp_scores / exp_scores.sum()

    predicted_index = int(np.argmax(probabilities))
    relation = LABELS[predicted_index]

    # Provisional strength estimate.
    strength = (
        0.0
        if relation == "neither"
        else round(float(probabilities[predicted_index]), 3)
    )

    return {
        "relation": relation,
        "strength": strength
    }


# Test the function
memory = "The user prefers Python over Java."
new_information = "The user now prefers Java over Python."

print(classify_evidence(memory, new_information))

{'relation': 'contradiction', 'strength': 0.999}


In [13]:
for memory, new_info, expected in tests:
    result = classify_evidence(memory, new_info)

    print("Expected:", expected)
    print("Result:  ", result)
    print("Correct: ", result["relation"] == expected)
    print("-" * 40)

Expected: contradiction
Result:   {'relation': 'contradiction', 'strength': 0.999}
Correct:  True
----------------------------------------
Expected: support
Result:   {'relation': 'support', 'strength': 0.992}
Correct:  True
----------------------------------------
Expected: neither
Result:   {'relation': 'neither', 'strength': 0.0}
Correct:  True
----------------------------------------


In [14]:

from datetime import datetime, timezone
import uuid

from llm_client import classify_evidence


def collect_evidence(memory, new_information):
    classification = classify_evidence(
        memory["content"],
        new_information
    )

    evidence = {
        "evidence_id": f"ev_{uuid.uuid4().hex[:8]}",
        "target_memory_id": memory["memory_id"],
        "new_information": new_information,
        "relation": classification["relation"],
        "strength": classification["strength"],
        "timestamp": datetime.now(timezone.utc).isoformat()
    }

    return evidence


ModuleNotFoundError: No module named 'llm_client'

In [15]:

%%writefile llm_client.py
import numpy as np

LABELS = ["contradiction", "support", "neither"]

def classify_evidence(memory_content, new_information):
    scores = nli_model.predict(
        [(new_information, memory_content)]
    )[0]

    scores = np.asarray(scores, dtype=float)

    exp_scores = np.exp(scores - np.max(scores))
    probabilities = exp_scores / exp_scores.sum()

    predicted_index = int(np.argmax(probabilities))
    relation = LABELS[predicted_index]

    strength = (
        0.0
        if relation == "neither"
        else round(float(probabilities[predicted_index]), 3)
    )

    return {
        "relation": relation,
        "strength": strength
    }


Writing llm_client.py


In [16]:

%%writefile evidence_collector.py
from datetime import datetime, timezone
import uuid

from llm_client import classify_evidence


def collect_evidence(memory, new_information):
    classification = classify_evidence(
        memory["content"],
        new_information
    )

    return {
        "evidence_id": f"ev_{uuid.uuid4().hex[:8]}",
        "target_memory_id": memory["memory_id"],
        "new_information": new_information,
        "relation": classification["relation"],
        "strength": classification["strength"],
        "timestamp": datetime.now(timezone.utc).isoformat()
    }


Writing evidence_collector.py


In [17]:

from evidence_collector import collect_evidence

memory = {
    "memory_id": "mem_0001",
    "content": "The user prefers Python over Java"
}

new_information = "The user now prefers Java over Python"

result = collect_evidence(memory, new_information)

print("Generated Evidence Object:")
print(result)

assert result["target_memory_id"] == "mem_0001"
assert result["relation"] in {"support", "contradiction", "neither"}
assert 0.0 <= result["strength"] <= 1.0

print("\nEvidence object validation passed!")


NameError: name 'nli_model' is not defined

In [18]:

%%writefile llm_client.py
import numpy as np
from sentence_transformers import CrossEncoder

MODEL_NAME = "cross-encoder/nli-deberta-v3-small"

# Load the model when this module is imported.
nli_model = CrossEncoder(MODEL_NAME)

LABELS = ["contradiction", "support", "neither"]


def classify_evidence(memory_content, new_information):
    scores = nli_model.predict(
        [(new_information, memory_content)]
    )[0]

    scores = np.asarray(scores, dtype=float)

    # Convert raw scores into probabilities.
    exp_scores = np.exp(scores - np.max(scores))
    probabilities = exp_scores / exp_scores.sum()

    predicted_index = int(np.argmax(probabilities))
    relation = LABELS[predicted_index]

    # Preliminary confidence estimate.
    strength = (
        0.0
        if relation == "neither"
        else round(float(probabilities[predicted_index]), 3)
    )

    return {
        "relation": relation,
        "strength": strength
    }


Overwriting llm_client.py


In [19]:
import importlib
import llm_client

importlib.reload(llm_client)

from evidence_collector import collect_evidence

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

In [20]:
memory = {
    "memory_id": "mem_0001",
    "content": "The user prefers Python over Java"
}

new_information = "The user now prefers Java over Python"

result = collect_evidence(memory, new_information)

print("Generated Evidence Object:")
print(result)

assert result["target_memory_id"] == "mem_0001"
assert result["relation"] in {"support", "contradiction", "neither"}
assert 0.0 <= result["strength"] <= 1.0

print("\nEvidence object validation passed!")

Generated Evidence Object:
{'evidence_id': 'ev_59d9bc7f', 'target_memory_id': 'mem_0001', 'new_information': 'The user now prefers Java over Python', 'relation': 'contradiction', 'strength': 0.999, 'timestamp': '2026-10-10T20:15:33.113568+00:00'}

Evidence object validation passed!


In [21]:
#examples

test_cases = [
    {
        "memory": "The user prefers Python.",
        "new_info": "The user still prefers Python.",
        "expected": "support"
    },
    {
        "memory": "The user prefers Python.",
        "new_info": "The user now prefers Java instead.",
        "expected": "contradiction"
    },
    {
        "memory": "The user prefers Python.",
        "new_info": "The user bought a new laptop yesterday.",
        "expected": "neither"
    },
    {
        "memory": "The library closes at 6 PM.",
        "new_info": "The library closes at 8 PM.",
        "expected": "contradiction"
    },
    {
        "memory": "The library closes at 6 PM.",
        "new_info": "The library closes at 6 PM.",
        "expected": "support"
    },
    {
        "memory": "The user lives in Bengaluru.",
        "new_info": "The user lives in Mumbai.",
        "expected": "contradiction"
    },
    {
        "memory": "The user owns a bicycle.",
        "new_info": "The user owns a bicycle.",
        "expected": "support"
    },
    {
        "memory": "The user studies computer science.",
        "new_info": "The user enjoys cooking.",
        "expected": "neither"
    },
    {
        "memory": "The meeting is on Monday.",
        "new_info": "The meeting is on Tuesday instead.",
        "expected": "contradiction"
    },
    {
        "memory": "The user prefers tea.",
        "new_info": "The user likes tea.",
        "expected": "support"
    }
]

correct = 0

for i, case in enumerate(test_cases, start=1):
    memory = {
        "memory_id": f"mem_test_{i:03d}",
        "content": case["memory"]
    }

    result = collect_evidence(memory, case["new_info"])
    passed = result["relation"] == case["expected"]
    correct += int(passed)

    print(f"\nTest {i}: {'PASS' if passed else 'FAIL'}")
    print("Expected:", case["expected"])
    print("Predicted:", result["relation"])
    print("Strength:", result["strength"])

print("\n" + "=" * 35)
print(f"Accuracy: {correct}/{len(test_cases)} "
      f"({correct / len(test_cases) * 100:.1f}%)")



Test 1: PASS
Expected: support
Predicted: support
Strength: 0.997

Test 2: PASS
Expected: contradiction
Predicted: contradiction
Strength: 0.998

Test 3: PASS
Expected: neither
Predicted: neither
Strength: 0.0

Test 4: PASS
Expected: contradiction
Predicted: contradiction
Strength: 0.999

Test 5: PASS
Expected: support
Predicted: support
Strength: 0.995

Test 6: PASS
Expected: contradiction
Predicted: contradiction
Strength: 0.999

Test 7: PASS
Expected: support
Predicted: support
Strength: 0.996

Test 8: FAIL
Expected: neither
Predicted: contradiction
Strength: 1.0

Test 9: PASS
Expected: contradiction
Predicted: contradiction
Strength: 0.994

Test 10: PASS
Expected: support
Predicted: support
Strength: 0.997

Accuracy: 9/10 (90.0%)


In [22]:

import numpy as np

memory = "The user studies computer science."
new_info = "The user enjoys cooking."

pairs = [
    (new_info, memory),
    (memory, new_info)
]

scores = nli_model.predict(pairs)
labels = ["contradiction", "support", "neither"]

for i, row in enumerate(scores):
    row = np.asarray(row, dtype=float)
    exp_scores = np.exp(row - np.max(row))
    probs = exp_scores / exp_scores.sum()

    print("\nOrder:", "New info -> Memory" if i == 0
          else "Memory -> New info")
    print(dict(zip(labels, np.round(probs, 4))))



Order: New info -> Memory
{'contradiction': np.float64(0.9995), 'support': np.float64(0.0), 'neither': np.float64(0.0005)}

Order: Memory -> New info
{'contradiction': np.float64(0.9969), 'support': np.float64(0.0001), 'neither': np.float64(0.0031)}


In [23]:

print("Model label mapping:")
print(nli_model.model.config.id2label)

print("\nModel label-to-ID mapping:")
print(nli_model.model.config.label2id)

print("\nRaw scores for the failed example:")
print(nli_model.predict([
    (
        "The user enjoys cooking.",
        "The user studies computer science."
    )
]))


Model label mapping:
{0: 'contradiction', 1: 'entailment', 2: 'neutral'}

Model label-to-ID mapping:
{'contradiction': 0, 'entailment': 1, 'neutral': 2}

Raw scores for the failed example:
[[ 6.1773124 -4.9303765 -1.4548066]]


In [24]:

!pip -q install -U transformers

from transformers import pipeline

alternative_nli = pipeline(
    "text-classification",
    model="cross-encoder/nli-roberta-base"
)

print("Alternative NLI model loaded!")


config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.56M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/1.01k [00:00<?, ?B/s]

Alternative NLI model loaded!


In [25]:

memory = "The user studies computer science."
new_info = "The user enjoys cooking."

result = alternative_nli(
    f"{new_info} [SEP] {memory}",
    truncation=True
)

print(result)


[{'label': 'contradiction', 'score': 0.6558711528778076}]
